# Purpose:
- Some czstack ROIs do not have a landmark at all. (the ones with manual coreg)
- How come?

In [7]:
from pathlib import Path
import os
import zarr
import numpy as np
import tifffile as tiff
import pandas as pd

data_path = Path('/root/capsule/data/')
code_path = Path('/root/capsule/code/')
mouse_id = 767018
coreg_dir = code_path / f'{mouse_id}_coreg'

In [8]:
czstack_centroid_fn = next(code_path.rglob(f'{mouse_id}_czstack_cell_centroids.csv'))
czstack_cell_centroids_df = pd.read_csv(czstack_centroid_fn).set_index('czstack_cell_id',drop=True)

czstack_seg_data_asset = [f for f in np.sort(os.listdir(data_path)) if f'multiplane-ophys_{mouse_id}' in f and 'cortical-zstack-seg' in f][0]
czstack_masks_file = data_path / czstack_seg_data_asset / 'channel_0_ref_0/segmentation_masks.tif'
czstack_masks = tiff.imread(czstack_masks_file)

In [20]:
# find a mask that includes the centroid
def find_mask_id(centroid, masks):
    x, y, z = np.round(centroid).astype(int)
    if (x < 0) or (y < 0) or (z < 0) or (z >= masks.shape[0]) or (y >= masks.shape[1]) or (x >= masks.shape[2]):
        return -1
    mask_id = masks[z, y, x]
    return mask_id


In [30]:
current_iter = 3
landmarks_file = next(coreg_dir.glob(f'{mouse_id}_landmarks_matched_ext_iter{current_iter}.csv'))
landmarks = pd.read_csv(landmarks_file, header=None)
columns = ['ids', 'active', 'czstack_x', 'czstack_y', 'czstack_z', 'hcr_x', 'hcr_y', 'hcr_z']
landmarks.columns = columns

def _get_zstack_id(x):
    return int(x.split('-')[0][2:]) if x.startswith('cz') else -1
def _get_hcr_id(x):
    return int(x.split('-')[1][3:]) if x.startswith('cz') else -1

landmarks['czstack_id'] = landmarks['ids'].apply(_get_zstack_id)
landmarks['hcr_id'] = landmarks['ids'].apply(_get_hcr_id)

In [31]:
landmarks

,ids,active,czstack_x,czstack_y,czstack_z,hcr_x,hcr_y,hcr_z,czstack_id,hcr_id
0,Pt-0,True,324.500000,5.481481,281.425926,1060.524232,1499.531938,1126.0,-1,-1
1,Pt-1,False,391.944204,190.823041,269.815818,908.597970,1264.736805,1096.0,-1,-1
2,cz365-hcr43757,True,300.778947,471.236842,169.500877,945.099734,856.311659,781.0,365,43757
3,cz248-hcr22386,True,487.488546,111.375000,124.553785,814.877224,1362.403688,578.0,248,22386
4,cz674-hcr102764,False,428.426422,61.203573,313.717442,874.069274,1442.312956,1231.0,674,102764
...,...,...,...,...,...,...,...,...,...,...
786,cz0676-hcr103596,True,359.416497,146.206721,312.518669,945.099734,1315.050048,1239.0,676,103596
787,cz0677-hcr105352,True,135.873409,183.008944,315.384245,1223.302370,1214.423563,1260.0,677,105352
788,cz0678-hcr102360,True,475.766031,205.534564,315.363779,782.321596,1240.073451,1227.0,678,102360
789,cz0682-hcr104705,True,101.245107,259.910142,316.977758,1240.073451,1113.797077,1252.0,682,104705


In [18]:
manual_landmarks = landmarks[landmarks['ids'].str.startswith('Pt-')]
print(len(manual_landmarks))
manual_landmarks

6


,ids,active,czstack_x,czstack_y,czstack_z,hcr_x,hcr_y,hcr_z
0,Pt-0,True,324.500000,5.481481,281.425926,1060.524232,1499.531938,1126.000000
1,Pt-1,False,391.944204,190.823041,269.815818,908.597970,1264.736805,1096.000000
226,Pt-806,True,65.988776,116.591277,274.363539,1328.934343,1289.424264,1117.212415
227,Pt-808,True,391.852482,151.214172,45.605979,932.275504,1324.427081,297.656733
228,Pt-816,True,483.479430,403.515004,44.274563,797.209641,945.554563,305.925948
229,Pt-817,True,455.503668,472.383220,32.569388,812.626915,870.173731,321.899945


In [37]:
i = 5
centroid = manual_landmarks.iloc[i][['czstack_x', 'czstack_y', 'czstack_z']].values.astype(float)
mask_id = find_mask_id(centroid, czstack_masks)
mask_centroid = landmarks.query(f'czstack_id == {mask_id}')[['czstack_x', 'czstack_y', 'czstack_z']].values
print(f'landmark centroid: {centroid}, mask_id: {mask_id}, mask centroid: {mask_centroid}')

landmark centroid: [455.50366822 472.38321958  32.56938759], mask_id: 1, mask centroid: [[455.80740038 471.73221063  32.0540797 ]]
